# DMPSP — Training on Kaggle (P100 16GB)

This notebook trains all three DMPSP components on Kaggle's free P100 GPU (30h/week).

**Setup**:
1. Upload your processed trajectories as a Kaggle dataset input
2. Add your API keys in Kaggle Secrets (Settings → Add-ons → Secrets)
3. Run all cells

**Expected training times on P100**:
- World model fine-tuning: ~8-10 hours (50K steps)
- Action proposal: ~6-8 hours (100K steps)
- Value function: ~4-6 hours (100K steps)

In [ ]:
# Install dependencies
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch-geometric==2.4.0',
    'transformers==4.44.0',
    'accelerate==0.33.0',
    'rdkit==2024.03.5',
    'httpx==0.27.2',
    'python-dotenv==1.0.1',
    'PyYAML==6.0.2',
], check=True)

print('Dependencies installed.')

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
# Adjust this path to wherever you cloned the repo on Kaggle
PROJECT_ROOT = Path('/kaggle/working/dmpsp')
sys.path.insert(0, str(PROJECT_ROOT))

# Data from Kaggle input dataset
DATA_DIR = Path('/kaggle/input/dmpsp-processed-data')

# Output checkpoints go to /kaggle/working
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Set API keys from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
# Optional — only needed if using live API scoring during training
# os.environ['ADMETLAB_API_KEY'] = secrets.get_secret('ADMETLAB_API_KEY')

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir: {DATA_DIR}')
print(f'Checkpoint dir: {CHECKPOINT_DIR}')
print(f'GPU available: {__import__("torch").cuda.is_available()}')

## Step 1: Verify data

In [ ]:
from data.preprocess import load_trajectories

train_trajs = load_trajectories(DATA_DIR / 'trajectories_train.pkl')
val_trajs = load_trajectories(DATA_DIR / 'trajectories_val.pkl')

print(f'Train trajectories: {len(train_trajs)}')
print(f'Val trajectories:   {len(val_trajs)}')
print(f'Example trajectory length: {len(train_trajs[0])}')
print(f'Example target SMILES: {train_trajs[0].states[0].target_smiles}')

## Step 2: Train World Model (ReactionT5 fine-tuning)

This is the heaviest step — fine-tuning ReactionT5 + property heads.
Gradient checkpointing is enabled by default for P100 16GB.

In [ ]:
world_model_ckpt = CHECKPOINT_DIR / 'world_model'

!python {PROJECT_ROOT}/train/train_world_model.py \
    --model_config {PROJECT_ROOT}/configs/model.yaml \
    --train_config {PROJECT_ROOT}/configs/train.yaml \
    --data_dir {DATA_DIR} \
    --out_dir {world_model_ckpt} \
    --device cuda \
    --phase dynamics \
    --batch_size 16 \
    --max_steps 50000 \
    --log_level INFO

## Step 3: Train Action Proposal Diffusion

In [ ]:
proposal_ckpt = CHECKPOINT_DIR / 'action_proposal'

!python {PROJECT_ROOT}/train/train_proposal.py \
    --model_config {PROJECT_ROOT}/configs/model.yaml \
    --train_config {PROJECT_ROOT}/configs/train.yaml \
    --data_dir {DATA_DIR} \
    --out_dir {proposal_ckpt} \
    --device cuda \
    --batch_size 32 \
    --max_steps 100000 \
    --log_level INFO

## Step 4: Train Value Function

In [ ]:
value_ckpt = CHECKPOINT_DIR / 'value_fn'

!python {PROJECT_ROOT}/train/train_value.py \
    --model_config {PROJECT_ROOT}/configs/model.yaml \
    --train_config {PROJECT_ROOT}/configs/train.yaml \
    --data_dir {DATA_DIR} \
    --out_dir {value_ckpt} \
    --device cuda \
    --batch_size 64 \
    --max_steps 100000 \
    --log_level INFO

## Step 5: Quick smoke test

In [ ]:
import json

# Run inference on aspirin
result = !python {PROJECT_ROOT}/scripts/plan_route.py \
    --smiles "CC(=O)Oc1ccccc1C(=O)O" \
    --weights_json '{{"yield":0.3,"cost":0.2,"safety":0.2,"manufacturability":0.15,"fto_risk":0.15}}' \
    --model_config {PROJECT_ROOT}/configs/model.yaml \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --device cuda \
    --max_steps 3

route = json.loads('\n'.join(result))
print(f"Route found: {route['n_steps']} steps")
print(f"Yield: {route['total_yield_fraction']:.3f}")
print(f"Objective scores: {route['objective_scores']}")
print(f"Planning time: {route['planning_time_seconds']:.2f}s")

## Saving checkpoints

Kaggle deletes `/kaggle/working` after session ends. Save checkpoints to a Kaggle dataset or download them.

In [ ]:
import shutil

# Zip checkpoints for download
shutil.make_archive('/kaggle/working/dmpsp_checkpoints', 'zip', str(CHECKPOINT_DIR))
print('Checkpoints zipped: /kaggle/working/dmpsp_checkpoints.zip')
print('Download from: Kaggle notebook → Output → Files')